<a href="https://colab.research.google.com/github/jclevitt1/neoantigen-pipeline/blob/stage-2a-and-interactive-viz/notebooks/option_A_component_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Option A — component test (stages 3→6) on curated real hotspots

Runs the scientifically interesting back half of the neoantigen pipeline
— candidate windows → **MHCflurry** presentation gate → **Łuksza** recognition
composite → string-of-beads construct — on a curated panel of **real** oncogenic
hotspot mutations (KRAS G12D/G12V, BRAF V600E, TP53 R175H, PIK3CA H1047R, EGFR L858R).

**Honesty:** these are real mutations on real human proteins, but this notebook makes
no claim any tumour carries them — it proves the *ranking science runs on real
mutations*, in isolation from the genomics front end. The full-tumour run is Option B.
See `docs/e2e_validation_notes.md`.

_First-run note: expect to iterate once (MHCflurry allele support, proteome download,
package paths). That's the point of the liveness run._

> ### Note on the outputs below
>
> **The saved outputs in this notebook predate a fix to stage 6** (the presentation-tier
> gate) and are kept as a record of the Colab run, not as the current result. The
> construct shown at the bottom has 20 peptides, 17 of them overlapping windows of the
> same KRAS G12D mutation that had been gated out as `low-presentation` and never
> scored — the bug that fix addressed.
>
> **For the current output, see [`docs/review/run_outputs/`](../docs/review/run_outputs/)**
> — the same test re-run on current code, where the construct is 7 peptides, all of
> them `tier=pass`. The *ranking* (stage 4) is unchanged between the two runs; only
> construct selection differs.


In [28]:
import sys, importlib, subprocess

def refresh(repo="/content/neoantigen_pipeline",
            pkgs=("NeoantigenVaccineConstructionPipeline", "core")):
    print(subprocess.run(["git", "-C", repo, "pull"], capture_output=True, text=True).stdout)
    for name in list(sys.modules):                       # evict cached modules
        if any(name == p or name.startswith(p + ".") for p in pkgs):
            del sys.modules[name]
    importlib.invalidate_caches()                        # see newly-added files/dirs
    print("refreshed — now RE-IMPORT the names you use")

refresh()
# CRITICAL: old bindings still point at old code, so re-import what you call:
from NeoantigenVaccineConstructionPipeline.demos.component_test import run_component_test

Updating c0e0829..7969ca3
Fast-forward
 .../stages/variants/fixture/curated.py                         | 10 +++++++---
 1 file changed, 7 insertions(+), 3 deletions(-)

refreshed — now RE-IMPORT the names you use


In [29]:
# 1. Get the repo onto Colab and on sys.path (it must contain core.py at its root).
#    Option A: mount Drive if the repo lives there.
# from google.colab import drive; drive.mount('/content/drive')
# REPO = '/content/drive/MyDrive/neoantigen_pipeline'
#    Option B: clone the private repo (needs a GitHub token with repo scope).
# !git clone https://<TOKEN>@github.com/jclevitt1/neoantigen-pipeline.git /content/neoantigen_pipeline
REPO = '/content/neoantigen_pipeline'
import sys; sys.path.insert(0, REPO)
print('repo on path:', REPO)

repo on path: /content/neoantigen_pipeline


In [30]:
# 2. Install MHCflurry + fetch the presentation models (offline, CPU-fine).
!pip -q install mhcflurry
!mhcflurry-downloads fetch models_class1_presentation

****************************************
The requested download 'models_class1_presentation' has already been downloaded. To re-download this data, first run: 
	rm -rf /root/.local/share/mhcflurry/4/2.2.0/models_class1_presentation/
in a shell and then re-run this command.
****************************************
Fetching 0/22 downloads from release 2.2.0
DOWNLOAD NAME                             ALREADY DOWNLOADED?    WILL DOWNLOAD NOW?    URL                  
models_class1_pan                         NO                     NO                    https://github.com/openvax/mhcflurry/releases/download/pre-2.0/models_class1_pan.selected.20200610.tar.bz2 
models_class1_presentation                YES                    NO                    https://github.com/openvax/mhcflurry/releases/download/pre-2.0/models_class1_presentation.20200611.tar.bz2 
models_class1_processing                  NO                     NO                    https://github.com/openvax/mhcflurry/releases/download/p

In [31]:
# 3. A real human proteome for the Łuksza dissimilarity-to-self term
#    (Swiss-Prot human, ~20k proteins — small).
!wget -qO /content/human.fasta.gz 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&compressed=true&query=organism_id:9606+AND+reviewed:true'
!gunzip -f /content/human.fasta.gz
!grep -c '^>' /content/human.fasta  # sanity: number of proteins

20431


In [32]:
# 4. Assemble + run stages 2a(fixture) → 2b(known HLA) → 3 → 4 → 5 → 6.
from NeoantigenVaccineConstructionPipeline.demos.component_test import run_component_test
out = run_component_test(workdir='/content/run_A', proteome='/content/human.fasta')
out

Plan: 2a-variants -> 2b-hla -> 3-candidates -> 4-rank -> 5-eval -> 6-construct
[curated-component-test] test 2a-variants       pass
[curated-component-test] test 2b-hla            pass
[curated-component-test] test 3-candidates      pass
[curated-component-test] test 4-rank            pass
[curated-component-test] test 5-eval            pass
[curated-component-test] test 6-construct       pass
[curated-component-test] 2a-variants            ran      0.0s
[curated-component-test] 2b-hla                 ran      0.0s
[curated-component-test] 3-candidates           ran      0.0s
Predicting processing.


100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


Predicting affinities.


100%|██████████| 6/6 [00:01<00:00,  4.40it/s]


Predicting processing.


...

100%|██████████| 1/1 [00:00<00:00, 29.83it/s]


Predicting affinities.


100%|██████████| 1/1 [00:00<00:00, 21.15it/s]


[curated-component-test] 4-rank                 ran      6.6s
[curated-component-test] 5-eval                 ran      1.3s
[curated-component-test] 6-construct            ran      0.0s


{'candidates_tsv': PosixPath('/content/run_A/candidates.tsv'),
 'ranked_tsv': PosixPath('/content/run_A/ranked.tsv'),
 'filtered_tsv': PosixPath('/content/run_A/filtered.tsv'),
 'construct_fasta': PosixPath('/content/run_A/construct.fasta'),
 'construct_json': PosixPath('/content/run_A/construct.json')}

In [33]:
# 5. Inspect the outputs: the ranked neoepitopes and the vaccine construct.
import pandas as pd
print('=== RANKED (top of stage 4) ===')
display(pd.read_csv(out['ranked_tsv'], sep='\t').head(20))
print('=== FILTERED (stage 5 survivors) ===')
display(pd.read_csv(out['filtered_tsv'], sep='\t'))
print('=== CONSTRUCT (stage 6 string-of-beads) ===')
print(out['construct_fasta'].read_text())

=== RANKED (top of stage 4) ===


,peptide,binding_affinity,immunogenicity,wt_peptide,tpm,gene,transcript,protein_change,length,vaf,best_allele,presentation_score,tier,agretopicity,dissimilarity_to_self
0,GADGVGKSAL,38.858508,3.116416,GAGGVGKSAL,NaN,KRAS,ENST00000311936,G12D,10,NaN,HLA-C*08:02,0.950041,pass,22.565355,1.0
1,ARHGGWTTKM,44.044977,2.202630,AHHGGWTTKM,NaN,PIK3CA,ENST00000263967,H1047R,10,NaN,HLA-C*07:01,0.838621,pass,9.048777,1.0
2,KLVVVGAVGV,52.254393,0.756890,KLVVVGAGGV,NaN,KRAS,ENST00000311936,G12V,10,NaN,HLA-A*02:01,0.715711,pass,2.131636,1.0
3,VVGAVGVGK,42.767601,0.257464,VVGAGGVGK,NaN,KRAS,ENST00000311936,G12V,9,NaN,HLA-A*11:01,0.911521,pass,1.293645,1.0
4,VVVGAVGVGK,35.826660,0.131166,VVVGAGGVGK,NaN,KRAS,ENST00000311936,G12V,10,NaN,HLA-A*11:01,0.958887,pass,1.140157,1.0
5,ITDFGRAKLL,95.048269,0.111660,ITDFGLAKLL,NaN,EGFR,ENST00000275493,L858R,10,NaN,HLA-C*08:02,0.732236,pass,1.118132,1.0
6,KIGDFGLATEK,52.901410,0.049394,KIGDFGLATVK,NaN,BRAF,ENST00000646891,V600E,11,NaN,HLA-A*11:01,0.857463,pass,1.050634,1.0
7,ITDFGRAKL,35.591786,-0.054785,ITDFGLAKL,NaN,EGFR,ENST00000275493,L858R,9,NaN,HLA-C*08:02,0.933284,pass,0.946689,1.0
8,VVVGADGVGK,46.959265,-0.139422,VVVGAGGVGK,NaN,KRAS,ENST00000311936,G12D,10,NaN,HLA-A*11:01,0.909714,pass,0.869861,1.0
9,KITDFGRAK,48.143277,-0.546934,KITDFGLAK,NaN,EGFR,ENST00000275493,L858R,9,NaN,HLA-A*11:01,0.850144,pass,0.578721,1.0


=== FILTERED (stage 5 survivors) ===


,peptide,autoimmunity_flag,clonality,manufacturability,binding_affinity,immunogenicity,wt_peptide,tpm,gene,transcript,protein_change,length,vaf,best_allele,presentation_score,tier,agretopicity,dissimilarity_to_self
0,GADGVGKSAL,False,unknown,pass,38.858508,3.116416,GAGGVGKSAL,NaN,KRAS,ENST00000311936,G12D,10,NaN,HLA-C*08:02,0.950041,pass,22.565355,1.0
1,ARHGGWTTKM,False,unknown,pass,44.044977,2.202630,AHHGGWTTKM,NaN,PIK3CA,ENST00000263967,H1047R,10,NaN,HLA-C*07:01,0.838621,pass,9.048777,1.0
2,KLVVVGAVGV,False,unknown,fail,52.254393,0.756890,KLVVVGAGGV,NaN,KRAS,ENST00000311936,G12V,10,NaN,HLA-A*02:01,0.715711,pass,2.131636,1.0
3,VVGAVGVGK,False,unknown,fail,42.767601,0.257464,VVGAGGVGK,NaN,KRAS,ENST00000311936,G12V,9,NaN,HLA-A*11:01,0.911521,pass,1.293645,1.0
4,VVVGAVGVGK,False,unknown,fail,35.826660,0.131166,VVVGAGGVGK,NaN,KRAS,ENST00000311936,G12V,10,NaN,HLA-A*11:01,0.958887,pass,1.140157,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,MNDARHGGWTT,False,unknown,pass,21491.042007,NaN,MNDAHHGGWTT,NaN,PIK3CA,ENST00000263967,H1047R,11,NaN,HLA-C*08:02,0.005859,low-presentation,NaN,NaN
224,NDARHGGWTTK,False,unknown,pass,8925.980037,NaN,NDAHHGGWTTK,NaN,PIK3CA,ENST00000263967,H1047R,11,NaN,HLA-A*11:01,0.021271,low-presentation,NaN,NaN
225,DARHGGWTTKM,False,unknown,pass,14407.116193,NaN,DAHHGGWTTKM,NaN,PIK3CA,ENST00000263967,H1047R,11,NaN,HLA-B*08:01,0.032070,low-presentation,NaN,NaN
226,ARHGGWTTKMD,False,unknown,pass,19162.185548,NaN,AHHGGWTTKMD,NaN,PIK3CA,ENST00000263967,H1047R,11,NaN,HLA-C*07:01,0.005440,low-presentation,NaN,NaN


=== CONSTRUCT (stage 6 string-of-beads) ===
>CURATED-DEMO_construct_aa peptides=20 linker=AAY
GADGVGKSALAAYARHGGWTTKMAAYITDFGRAKLLAAYKIGDFGLATEKAAYVGADGVGKAAYGADGVGKSAAYADGVGKSAAAYDGVGKSALAAYVVGADGVGKAAYVGADGVGKSAAYGADGVGKSAAAYADGVGKSALAAYDGVGKSALTAAYEYKLVVVGADAAYYKLVVVGADGAAYVVGADGVGKSAAYVGADGVGKSAAAYADGVGKSALTAAYDGVGKSALTIAAYTEYKLVVVGAD
>CURATED-DEMO_construct_nt codon_table=human_high_freq
GGCGCCGACGGCGTGGGCAAGAGCGCCCTGGCCGCCTACGCCCGCCACGGCGGCTGGACCACCAAGATGGCCGCCTACATCACCGACTTCGGCCGCGCCAAGCTGCTGGCCGCCTACAAGATCGGCGACTTCGGCCTGGCCACCGAGAAGGCCGCCTACGTGGGCGCCGACGGCGTGGGCAAGGCCGCCTACGGCGCCGACGGCGTGGGCAAGAGCGCCGCCTACGCCGACGGCGTGGGCAAGAGCGCCGCCGCCTACGACGGCGTGGGCAAGAGCGCCCTGGCCGCCTACGTGGTGGGCGCCGACGGCGTGGGCAAGGCCGCCTACGTGGGCGCCGACGGCGTGGGCAAGAGCGCCGCCTACGGCGCCGACGGCGTGGGCAAGAGCGCCGCCGCCTACGCCGACGGCGTGGGCAAGAGCGCCCTGGCCGCCTACGACGGCGTGGGCAAGAGCGCCCTGACCGCCGCCTACGAGTACAAGCTGGTGGTGGTGGGCGCCGACGCCGCCTACTACAAGCTGGTGGTGGTGGGCGCCGACGGCGCCGCCTACGTGGTGGGCGCCGACGGCGTGGGCAAGAGCGCCGCCTACGTGGGCGCCGACGGCG